# Edge-AUI Foundational Model Training & Edge Quantization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taofeeqhamzat/edge-aui-model-preparation/blob/explore/pipeline/revision/1/notebooks/Training.ipynb)

This notebook executes the end-to-end pipeline for the **Edge-AUI Framework** in hosted environments (Google Colab / Kaggle):
1. Synchronizes interaction datasets directly from the Hugging Face Hub (`T40/edge-aui-framework-data`).
2. Extracts 18-dimensional MicroTensors (9 kinematic features + 9 modality masks) from continuous behavioral trajectories.
3. Trains a lightweight PyTorch Gated Recurrent Unit (GRU) model with hardware acceleration (CUDA / MPS / CPU).
4. Exports the PyTorch model to dynamic ONNX and applies INT8 Post-Training Quantization (PTQ).
5. Empirically validates edge constraints: **Memory Footprint < 20MB** and **Inference Latency < 50ms**.

## 1. Hosted Environment Setup & Accelerator Detection
Automatically configures required dependencies, sets up repository paths, and inspects available GPU accelerators.

In [ ]:
# Safe autoreload (Python 3.13 removed legacy 'imp' module used by older IPython)
try:
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic('load_ext', 'autoreload')
        ip.run_line_magic('autoreload', '2')
except Exception:
    pass

import os
import sys
import torch

# Environment detection
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IN_COLAB:
    print("[Setup] Detected Google Colaboratory. Setting up environment...")
    if not os.path.exists("src") and not os.path.exists("../src"):
        !git clone -b explore/pipeline/revision/1 https://github.com/taofeeqhamzat/edge-aui-model-preparation.git
        %cd edge-aui-model-preparation
    else:
        # Sync latest commits if already cloned
        try:
            !git checkout explore/pipeline/revision/1
            !git pull origin explore/pipeline/revision/1
        except Exception:
            pass
    !pip install -q huggingface_hub datasets torch pandas numpy matplotlib seaborn onnx onnxruntime onnxscript scikit-learn tqdm
elif IN_KAGGLE:
    print("[Setup] Detected Kaggle Kernel. Setting up environment...")
    !pip install -q huggingface_hub datasets torch pandas numpy matplotlib seaborn onnx onnxruntime onnxscript scikit-learn tqdm
else:
    print("[Setup] Detected Local / Self-Hosted Environment.")

# Add src to sys.path across all possible working directory locations
for path_candidate in ["src", "../src", "./model-preparation/src"]:
    abs_p = os.path.abspath(path_candidate)
    if os.path.isdir(abs_p) and abs_p not in sys.path:
        sys.path.insert(0, abs_p)
        print(f"[System] Added to sys.path: {abs_p}")
        break

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else "cpu")
print(f"[Hardware] Active Execution Device: {device}")
if device.type == "cuda":
    print(f"[Hardware] GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"[Hardware] GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Hosted Data Synchronization (Hugging Face Hub)
Verifies local dataset cache or synchronizes from `T40/edge-aui-framework-data`.
> **Note on HTTP 429 Rate Limits:** Google Colab shares public IP addresses across many users. To avoid rate limits, set `HF_TOKEN` in Colab Secrets (🔑). `ensure_dataset()` will automatically throttle concurrent requests and fall back to single-stream Git clone if rate-limited.

In [ ]:
from data_manager import ensure_dataset, get_hf_token, is_colab

# Check Hugging Face authentication token
token = get_hf_token()
if token is None and is_colab():
    print("ℹ️ TIP: To ensure maximum rate limits, consider setting HF_TOKEN in Colab Secrets (🔑).")

# Synchronize datasets (handles rate limits, throttles requests, and auto-falls back to Git stream sync)
DATA_ROOT = ensure_dataset(repo_id="T40/edge-aui-framework-data", token=token)
print(f"[Data] Datasets verified at: {DATA_ROOT}")

## 3. Dataset Loading & Feature Inspection
Extracts 500ms sliding windows and constructs temporal sequences (`seq_len=8`).

In [ ]:
import importlib
import preprocessing
import training
importlib.reload(preprocessing)
importlib.reload(training)
from training import load_foundation_dataset
from preprocessing import FEATURE_NAMES, LABEL_MAP, MICROTENSOR_DIM

# Load sequence dataset
dataset = load_foundation_dataset(data_dir=DATA_ROOT, max_sequences=1000, split='train')
print(f"[Dataset] Total behavioral sequences extracted: {len(dataset)}")

if len(dataset) > 0:
    sample_x, sample_y = dataset[0]
    print(f"[Dataset] Input Tensor Shape (seq_len, features): {sample_x.shape}")
    print(f"[Dataset] Target Outcome Label: {sample_y.item()}")
    print(f"[Dataset] MicroTensor Dimension: {MICROTENSOR_DIM} ({len(FEATURE_NAMES)} features + {len(FEATURE_NAMES)} masks)")
    print("[Dataset] Outcome Classes:", LABEL_MAP)

## 4. Train Foundational GRU Model
Executes the recurrent training loop, learning cross-domain motor dynamics and structural interaction patterns.

In [ ]:
from training import train_foundation_model
import matplotlib.pyplot as plt

# Train model with empty-class safe smoothed loss weighting and validation tracking
results = train_foundation_model(
    data_dir=DATA_ROOT,
    epochs=5,
    batch_size=64,
    lr=1e-3,
    class_weights="smoothed",
    device=str(device),
    verbose=True
)

history = results["history"]
epochs_range = range(1, len(history["loss"]) + 1)

# Plot Training & Validation Loss and Accuracy Metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(epochs_range, history["loss"], marker='o', color='crimson', label='Train Loss')
if history.get("val_loss"):
    ax1.plot(epochs_range, history["val_loss"], marker='x', color='darkorange', linestyle='--', label='Val Loss')
ax1.set_title("CrossEntropy Loss Progression (Empty-Class Safe Weighting)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(epochs_range, history["accuracy"], marker='s', color='navy', label='Train Accuracy')
if history.get("val_accuracy"):
    ax2.plot(epochs_range, history["val_accuracy"], marker='^', color='teal', linestyle='--', label='Val Accuracy')
ax2.set_title("Classification Accuracy Progression")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 5. Comprehensive Classification Evaluation & Baseline Comparison
Evaluates the converged `EdgeAUIGRU` model against the AdSERP validation partition:
- **Primary Metrics:** Macro-F1 (unweighted across classes with support), Weighted-F1, per-class Precision/Recall/F1, Confusion Matrix, and Majority-Class baseline comparison.
- **Secondary Ranking Diagnostics:** Hit Rate@1 (Top-1 accuracy), Hit Rate@3, and Mean Reciprocal Rank (MRR), evaluated as auxiliary diagnostics rather than primary classification criteria.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from training import OUTCOME_TAXONOMY

val_eval = results.get("val_evaluation")
if val_eval is not None:
    cm = np.array(val_eval["confusion_matrix"])
    class_names = [OUTCOME_TAXONOMY.get(i, str(i)) for i in range(len(cm))]
    
    # Visual Confusion Matrix Heatmap
    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.title("Confusion Matrix: Observable Downstream Outcomes (Validation Split)")
    plt.xlabel("Predicted Outcome")
    plt.ylabel("True Outcome")
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    print(f"Validation Macro-F1:    {val_eval['macro_f1']:.4f}")
    print(f"Validation Weighted-F1: {val_eval['weighted_f1']:.4f}")
    print(f"Validation Accuracy:    {val_eval['accuracy']:.2f}%")
    
    ranking = val_eval["ranking_diagnostics"]
    print(f"Secondary HR@1:         {ranking['hr@1'] * 100:.2f}%")
    print(f"Secondary HR@3:         {ranking['hr@3'] * 100:.2f}%")
    print(f"Secondary MRR:          {ranking['mrr']:.4f}")
    
    base = val_eval.get("majority_baseline")
    if base is not None:
        delta = val_eval['macro_f1'] - base['macro_f1']
        print(f"Majority Baseline Macro-F1: {base['macro_f1']:.4f} (Delta: {'+' if delta >= 0 else ''}{delta:.4f})")
else:
    print("Validation evaluation was not executed.")

## 6. TargetInterventionHead Context Interface Verification
Verifies the modular `TargetInterventionHead` architectural interface `[h_T, context_vector] -> 5 actions` (ADR-003).
The target UI context vector $C \in \mathbb{R}^6$ conditions intervention decisions based on local viewport state and active element properties.
*(Policy fine-tuning is deferred to Phase 5 once target testbed telemetry is collected).*

In [ ]:
import torch
from training import EdgeAUIGRU, TargetInterventionHead

# Instantiate target intervention head with context conditioning (ADR-003)
context_head = TargetInterventionHead(hidden_dim=64, context_dim=6, num_classes=5)
dummy_latent = torch.randn(4, 64)
dummy_context = torch.randn(4, 6)

# Forward pass conditioning on both latent representation h_T and UI context
intervention_logits = context_head(dummy_latent, dummy_context)
print(f"TargetInterventionHead Output Shape: {intervention_logits.shape} (Batch=4, Interventions=5)")
assert intervention_logits.shape == (4, 5), "Dimension mismatch on conditioned projection head!"
print("TargetInterventionHead context conditioning interface verified successfully.")

## 7. Dynamic ONNX Export & INT8 Quantization
Compiles the PyTorch graph to ONNX format and applies dynamic Post-Training Quantization (QUInt8) to compress the model weights.

In [ ]:
from export import export_and_quantize

metrics = export_and_quantize(
    model_path=results["model_path"],
    output_dir="models"
)

## 8. Edge Constraint Verification
Empirically asserts strict edge-native deployment constraints:
- **Memory Footprint:** $\le 20\text{ MB}$
- **Inference Latency:** $\le 50\text{ ms}$ (zero UI jank threshold)

In [ ]:
print("=" * 50)
print("       EDGE ARCHITECTURE BENCHMARK REPORT")
print("=" * 50)
print(f"Quantized Model Path:    {metrics['quantized_onnx_path']}")
print(f"Storage Footprint:       {metrics['file_size_mb']:.4f} MB  (Threshold: < 20.0 MB) -> {'PASS' if metrics['file_size_mb'] < 20 else 'FAIL'}")
print(f"Average Latency (CPU):   {metrics['avg_latency_ms']:.4f} ms  (Threshold: < 50.0 ms) -> {'PASS' if metrics['avg_latency_ms'] < 50 else 'FAIL'}")
print(f"P95 Latency (CPU):       {metrics['p95_latency_ms']:.4f} ms")
print("=" * 50)

assert metrics['file_size_mb'] < 20.0, "Memory constraint exceeded!"
assert metrics['avg_latency_ms'] < 50.0, "Latency constraint exceeded!"
print("All Edge-AUI production deployment constraints passed!")

## 9. Artifact Download (Colab / Local Export)
Provides one-click download of the quantized ONNX artifact (`model_int8.onnx`) for direct integration with the browser runtime.

In [ ]:
if IN_COLAB:
    from google.colab import files
    print("Triggering browser download of quantized ONNX model...")
    files.download(metrics['quantized_onnx_path'])
else:
    print(f"Model artifact is ready locally at: {os.path.abspath(metrics['quantized_onnx_path'])}")